<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
</head>
<body>
    <div style="display: flex; align-items: center;">
        <div>
            <h1>TD 10 - Sequential Perception and Review</h1>
            <h2>Understanding human behavior with cognitive models</h2>
            <h3>Master in Cognitive Science</h3>
            <h4>École Normale Supérieure - PSL</h4>
            <p> Valentin Wyart - Lecturer<br>
                Amric Trudel - Practical Sessions (TD)<br>
                Notebook author: <a href="mailto:amric.trudel@ens.psl.eu">amric.trudel@ens.psl.eu</a></p>
        </div>
        <div>
            <img src="images/logo_ens.png" style="height: 70px; margin-left: 10px;" />
        </div>
    </div>
</body>
</html>

## Objective
In this TD, we will work on a new type of task: a **sequential perception task**.

You will implement two different models and run all the steps that we have seen so far:
- Understand the task
- Implement two models
- Simulate behavior
- Parameter recovery
- Model recovery
- Fit models to behavioral data
- Model selection:
    - Fixed effect
    - Random effect

In [3]:
from typing import Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import seaborn as sns
from tqdm import trange, tqdm

from tests import test_integration_model_update, test_integration_model_policy, test_extremum_model_policy, \
    test_extremum_model_update
from models import Model, Parameter
from sequential_perception_task import SequentialPerceptionTask, dist_orange, dist_blue

## The Task

We will work on a sequential perception task in which the subject must determine from which of two bags (orange or blue) the stimuli come are drawn. For each block, the steps are the following:
1. For a given block, one of the two bags is used to draw beads from.
2. The subject is shown a sequence of stimuli (colored beads) sampled from this bag.
3. The subject must choose which bag is most likely to have produced the observed sequence of stimuli.

The color of each bead can be any shade along a continuous gradient from blue to orange. The figure below illustrates how colors are distributed for each bag.

<img src="images/task_stimuli_distributions.png" alt="Task" style="width: 500px;"/>

The stimuli presented are circles, each with a color represented by a number between -1 and 1:
   - -1 corresponds to completely blue,
   - 0 is grey,
   - 1 is completely orange.

Both bags contain beads of all colors, but the orange bag contains more orange beads, while the blue bag has more blue beads.

The `dist_orange` and `dist_blue` distributions, imported in this notebook,represent the color distributions for each bag.
They are implemented to behave like `scipy.stats` distributions. Two useful methods for these distributions are:
- `<distribution>.pdf(x)`: **Probability density function**, which returns the likelihood that a stimulus of color `x` was drawn from that distribution.
- `<distribution>.rvs(size)`: **Random variates**, which generates a sample of `size` values draw from the distribution.

📝Plot the probability density functions of those two distributions on the same graph. You should obtain something like a merged version of the figure above.

In [ ]:
# x: Generate a sequence of 100 color levels, covering the whole range
x = np.linspace(..., ..., 100)

# y: For each distribution, get the probabilities associated with all color levels
p_orange = ...
p_blue = ...

# Plot both probability distributions on the same graph
...

The `SequentialPerceptionTask` class is already provided and imported in this notebook. It functions similarly to the `BanditTask` class used in previous TDs, but we need to adapt the notions of *trial* and *block*. In this experiment, we will consider that a **trial** corresponds to the presentation of a new stimulus, and a **block** corresponds to the sequence of stimuli presented to the subject before they choose a bag.

In other words:
- `n_trials` represent the number of stimuli shown in each sequence
- The subject must choose a bag after the n trials
- `n_blocks` is the number of sequences (decisions) taken in the experiment

⚠️This task is different from the Bandit Task. Now we have only one subject response per block, and there are **no rewards**.

⚙️ Run the cell below to instantiate a `SequentialPerceptionTask` with **8 trials** and **8 blocks.** We use a **seed of 42** to make the task reproducible.

The `plot()` method is then called to visualize the task. In the plot, you should see 8 rows, each representing a block (sequence) of stimuli, followed by a final circle at the end of each row, indicating the actual bag from which that sequence was drawn (ground truth).

In [ ]:
demo_task = SequentialPerceptionTask(n_trials=8, n_blocks=8, seed=42)
demo_task.plot()

## The Models
We will propose two models to explain the subject's behavior in this task:
1. An **integration model** that accumulates evidence over time
2. An **extremum detection model** that only remembers the most salient stimulus

Before you start this sections, pay careful attention to the following points:

**The `Model` base class**

Both models inherit from the `Model` base class, which is written in the `models.py` file. This base class provides several useful methods that you are now familiar with and that you will **not** need to re-implement today:
 - `simulate()`: Simulates the task with the model and returns the choices.
- `fit()`: Fits the model to the data and returns the fitted model.
- `log_likelihood()`: Computes the log likelihood of the model given the behavioral data.
- `plot()`: Plots the model's behavior on the task.

**The Parameters**

Another new feature in this TD is that each model class has a `parameters` attribute, a list of `Parameter` objects (also defined in the `models.py` file). Each `Parameter` has the following attributes, which will be useful for model fitting with BADS:
- `name`: Parameter name.
- `initial_value`: Initial value to use in optimization.
- `distribution`: A `scipy.stats` distribution object specifying the parameter's prior distribution.
- `bounds`: Tuple specifying the minimum and maximum values.
- `plausible_bounds`: Tuple specifying the range within which the parameter is expected to fall with 90% probability.

**Vectorized implementation**

To efficiently compute the log likelihood, we will use vectorized processing: all blocks will be processed simultaneously. This means NumPy array must be two-dimensional:
- The first dimension (axis 0) corresponds to the **blocks**
- The second dimension (axis 1) corresponds to the **trials**

So:
- **stimuli** will be a 2D numpy array of shape (n_blocks, n_trials)
- **stimulus** (one trial) will be a 2D numpy array of shape (n_blocks, 1)
- **probability of choice** will be a 2D numpy array of shape (n_blocks, 1)
- **evidence** will be a 2D numpy array of shape (n_blocks, 1)

Handling vectorized operations can be very tricky when you are not used to it, so don't hesitate to ask for help to get started.

### Model 1: Integration model
The **integration model** accumulates evidence over time. At the end of each block, it uses a softmax policy to choose the bag based on the total accumulated evidence.

The `IntegrationModel` class implements this model and has two parameters:
- `leak`: Controls how much the evidence is forgotten over time. A value of 0 means no leak (the model remembers everything), while a value of 1 means complete leak (the model forgets everything).
- `temperature`: Controls the stochasticity of the model. A value of 0 means the model always chooses the bag with the highest evidence (deterministic), and a value of 1 means high randomness (same as usual).

Instructions for completing this class are provided below the declaration (scroll👇).

In [ ]:
# READ INSTRUCTIONS BELOW
class IntegrationModel(Model):

    ## 1) TO DO: Define the parameter space
    parameters = [
        Parameter(
            name='leak',
            initial_value=...,
            distribution=scipy.stats.uniform(0, 1),
            bounds=(..., ...),
            plausible_bounds=(..., ...)
        ),
        Parameter(
            name='temperature',
            initial_value=...,
            distribution=scipy.stats.expon(scale=0.1),
            bounds=(..., ...),
            plausible_bounds=(..., ...)
        )
    ]

    def __init__(self, leak: float, temperature: float):
        self.leak: float = leak
        self.temperature: float = temperature
        self.n_blocks: Optional[int] = None
        self.evidence: Optional[np.ndarray[float]] = None

    ## 2) TO DO: Implement the update method
    def update(self, stimulus):
        # Get the pfd density of the stimulus for each distribution
        p_orange = ...
        p_blue = ...

        # Compute the log odds ratio
        log_odds_ratio = ...

        # Update the evidence with the leak
        self.evidence = ...

    ## 3) TO DO: Implement the policy method
    def policy(self):
        # Softmax policy to compute probability of choosing 'orange'
        prob_choose_orange = ...

        return prob_choose_orange


    def reset(self, n_blocks: int):
        self.n_blocks = n_blocks
        self.evidence = np.zeros((n_blocks, 1))

    def __repr__(self):
        return f"IntegrationModel(lk={self.leak: .2f}, t={self.temperature: .2f})"

#### 1) Parameter specifications
📝 In the `IntegrationModel` class, define the parameter specifications using values that you think are the most reasonable (or even better, discuss it with your neighbor!)

⚙️ You can run the `plot_param_specs()` method to visualize the parameter space that you created (run the cell below).

In [ ]:
IntegrationModel(0, 0).plot_param_specs()

#### 2) Updating evidence
The evidence starts at 0 and is updated after each trial. When the evidence is positive, the model favors the orange bag, and when it gets negative, the model tilts towards the blue bag.

The evidence is calculated as the **log odds ratio** of the two distributions:
$$
\text{evidence} = \log \left( \frac{p_{\text{orange}}}{p_{\text{blue}}} \right)
$$
Where:
- $p_{\text{orange}}$ is the probability density of the orange bag for the current stimulus
- $p_{\text{blue}}$ is the probability density of the blue bag for the current stimulus
- $\frac{p_{\text{orange}}}{p_{\text{blue}}}$ is the odds ratio between the two distributions

📝Implement the `update()` method to update the evidence using the log odds ratio, and make sure to include the effect of the leak parameter (I let you think about how to do this).

⚙️ You can test your implementation by running the cell below.

In [ ]:
test_integration_model_update(IntegrationModel)

#### 3) Policy
The model uses a softmax policy to decide which bag to choose based on the accumulated evidence. The probability of choosing the orange bag is:
$$
P(\text{orange}) = \frac{1}{1 + \exp(-\frac{\text{evidence}}{\text{temperature}})}
$$

📝Implement the `policy()` method to compute the **probability of choosing the orange bag** using the softmax policy. The temperature parameter controls the stochasticity of the model.

⚙️ You can test your implementation by running the cell below.

In [ ]:
test_integration_model_policy(IntegrationModel)

### Simulate the Integration model
Now that you have implemented the `IntegrationModel`, you can use it to simulate the task. Let's start with the optimal model, which is the one that has no leak and no stochasticity.

📝Instantiate an optimal model and call the simulate method (you pass it the `demo_task` object and set `plot=True`).
See how the evidence is updated with each stimulus.

In [ ]:
optimal_integration_model = ...
... # Simulate with the demo_task and visualize

📝Implement another `IntegrationModel` with different parameters and see if the evidence is updated differently and if the choices are different. Try to understand the effect of each parameter.

In [ ]:
# Your code here
other_integration_model = ...
...

### Model 2: Extremum detection model
For comparison, let's make a simpler (and a bit more stupid) model: the **extremum detection model**. This model only remembers the most salient stimulus in the sequence (i.e. the most intense shade of either orange or blue), and reports its color at the end. To spike things up a bit, the model will use an epsilon-greedy policy: with probability `epsilon`, it chooses the opposite bag.

**Salience** is defined as the absolute value of the stimulus (how far it is from 0).

In other words:
- If the most salient stimulus > 0 (looks orange):
    - Choose orange with probability `1 - epsilon`
    - Choose blue with probability `epsilon`
- If the most salient stimulus < 0 (looks blue):
    - Choose blue with probability `1 - epsilon`
    - Choose orange with probability `epsilon`

In [ ]:
class ExtremumDetectionModel(Model):

    ## 1) TO DO: Define the parameter space
    parameters = [
        Parameter(
            name='epsilon',
            initial_value=...,
            distribution=scipy.stats.uniform(0, 1),
            bounds=(..., ...),
            plausible_bounds=(..., ...)
        )
    ]

    def __init__(self, epsilon: float):
        self.epsilon: float = epsilon
        self.n_blocks: Optional[int] = None
        self.evidence: Optional[np.ndarray[float]] = None

    ## 2) TO DO: Implement the update method
    def update(self, stimulus):
        # Your code here

        self.evidence = ...

    ## 3) TO DO: Implement the policy method
    def policy(self):
        # Include the epsilon parameter to obtain the epsilon-greedy probability of choosing 'orange'
        p_orange = ...

        return p_orange

    def reset(self, n_blocks: int):
        self.n_blocks = n_blocks
        self.evidence = np.zeros((n_blocks, 1))

    def __repr__(self):
        return f'ExtremumDetectionModel(epsilon={self.epsilon: .2f})'

#### 1) Parameter specifications
📝 Define the parameter specifications using values that you think are the most reasonable.

⚙️ You can run the `plot_param_specs()` method to visualize the parameter space that you created (run the cell below).

In [ ]:
ExtremumDetectionModel(0).plot_param_specs()

#### 2) Updating evidence
In this model, the evidence will be stored as the value of the most salient stimulus.

At every trial, if the new stimulus is more salient (i.e. has a higher absolute value) than the current evidence, we update the evidence with the new stimulus.

📝 Implement the `update()` method to update the evidence with the most salient stimulus.

Run the unit tests in the cell below to test your implementation.

In [ ]:
test_extremum_model_update(ExtremumDetectionModel)

#### 3) Policy
The model uses an epsilon-greedy policy to choose the bag based on the most salient stimulus. The probability of choosing the orange bag is given by:
$$
P(\text{orange}) = \begin{cases}
1 - \epsilon & \text{if evidence} > 0 \\
\epsilon & \text{if evidence} < 0
\end{cases}
$$
📝Implement the `policy()` method to compute the probability of choosing the orange bag using the epsilon-greedy policy. The epsilon parameter controls the stochasticity of the model.

Run the unit tests in the cell below to test your implementation.

In [ ]:
test_extremum_model_policy(ExtremumDetectionModel)

### Simulate the Extremum detection model
Now that you have implemented the `ExtremumDetectionModel`, you can use it to simulate the tak. Let's start with the optimal model, i.e. with 0 stochasticity.

📝Implement an optimal `ExtremumDetectionModel` and simulate it with the `demo_task` object.

In [ ]:
optimal_extremum_model = ...
... # Simulate with the demo_task and visualize

📝Implement another `ExtremumDetectionModel` with a different `epsilon` and see the effect on the evidence and the responses.

In [ ]:
# Your code here
other_extremum_model = ...
...

## Define the experimental task

Our study will use a task that is much longer than the demo task we used previously.

The task use in the experiment has 8 trials and 200 blocks, and was seeded at 42.

In [ ]:
task = SequentialPerceptionTask(n_trials=8, n_blocks=200, seed=42)

## Parameter recovery
👉 You can skip this section if you are running out of time (i.e. if you have spent more than 1 hour so far). Go directly to Model recovery.

Before analyzing participant data, we need to ensure that the parameters of our models are identifiable. We will first write a generic parameter recovery function, and we will run it with both models.

### The parameter recovery loop
📝 Complete the `perform_parameter_recovery` function to perform parameter recovery for a model given task and model.
> Notes:
> - Replace all `...` with the appropriate code.
> - You can refer to TD6 for inspiration
> - To debug your function, it's better to run it with a small number of sampling (done for you at the bottom of the cell).

In [ ]:
def perform_parameter_recovery(task, ModelClass, n_samplings):
    results = []
    for i in trange(n_samplings):
        # Sample parameters and instantiate original model
        ## CAREFUL! This part has changed :)
        original_params = {
            parameter.name: ... # TODO: Sample a parameter value from its distribution
            for parameter in ModelClass.parameters
        }
        original_model = ... # TODO

        # Simulate task with the original model
        choices = ...  # TODO

        # Fit the model to the simulated data
        recovered_model = ModelClass.fit(task, ..., verbose=False)  # TODO

        # Get the recovered parameters (this part is done for you 🙂)
        recovered_params = {
            f"recovered_{parameter.name}": getattr(recovered_model, parameter.name)
            for parameter in ModelClass.parameters
        }
        # Pack the original and recovered parameters into a Pandas Series
        results.append(pd.Series({**original_params, **recovered_params}))

    # Transform the list of Series into a DataFrame
    return pd.concat(results, axis=1).transpose()

## Test
# We run the function with the demo task to test it
perform_parameter_recovery(demo_task, IntegrationModel, 2)
perform_parameter_recovery(demo_task, ExtremumDetectionModel, 2)
print("👌The code runs")

### Parameter recovery for Model 1 (Integration Model)
📝Run parameter recovery for the `IntegrationModel` with 30 samplings and store the result in `param_recov_integration`.

In [ ]:
param_recov_integration = ...
param_recov_integration.head()

The function that plots the confusion matrix for the parameter recovery results is provided to you 🎁

In [ ]:
def plot_param_recovery_confusion_matrix(param_recovery_results: pd.DataFrame):
    correlations = param_recovery_results.corr()
    row_names = [col_name for col_name in param_recovery_results.columns if 'recovered' in col_name]
    column_names = [col_name for col_name in param_recovery_results.columns if 'recovered' not in col_name]
    confusion_matrix = correlations.loc[row_names, column_names]

    plt.figure(figsize=(3, 2))
    sns.heatmap(confusion_matrix, center=0, annot=True, fmt='.2f', cmap='coolwarm')
    plt.show()


📝Plot the confusion matrix of the parameter recovery results of Model 1

In [ ]:
# Your code here

### Parameter recovery for Model 2 (Extremum detection)

📝 Run the parameter recovery for the `ExtremumDetectionModel` with 30 samplings.
> Note: you might get a warning when you run the code. You can ignore it.

In [ ]:
param_recov_extremum = ...
param_recov_extremum.head()

There is only one parameter in the `ExtremumDetectionModel`, so it doesn't really make sense to plot a confusion matrix.
How else could you visualize the results?

📝Make a graphical representation of the parameter recovery result of the `ExtremumDetectionModel` (Model 2).

In [ ]:
# Your code here

## Model Recovery


### Akaike Information Criterion
Rememeber, when we perform parameter reccovery, we have to compare the fitness of different models on the same behavioral data. We saw many ways of doing this, but in this TD we will use the **Akaike Information Criterion**.

As a reminder, the formula is the following:
$$
\text{AIC} = -2 \cdot \log L + 2k
$$
Where:
- $L$ is the likelihood of the behavior given the fitted model
- $k$ is the number of parameters in the model

📝 Complete the `akaike_information_criterion()` function.

In [ ]:
def akaike_information_criterion(model, task, choices) -> float:
    k = len(model.parameters)
    ll = ...
    aic = ...
    return aic

### Model recovery loop
👉 If you have less than 30 minutes left, skip this section and go directly to **Participant Data Analysis**

📝Complete the `perform_model_recovery` function to perform model recovery for a task and a list of models.
>Notes:
> - Replace all instances of `...` with appropriate code.
> - You can look at TD 7 for inspiration.
> - The way we sample the parameters has changed since last TDs. You should use the `.parameters` attribute of the model class, like we did in the `perform_parameter_recovery` function.
> - We will only use the Akaike Information Criterion (AIC) to select the winning model, so we will not bother comparing different criteria.

In [ ]:
def perform_model_recovery(task, model_classes, n_rounds):
    # Instantiate the results as a zero matrix with the model names as row and column labels
    model_names = [model_class.__name__ for model_class in model_classes]
    results = pd.DataFrame(data=0, columns=model_names, index=model_names)

    for _ in trange(n_rounds):

        for simul_idx, simul_model_class in enumerate(model_classes):
            ########################################################
            ### Simulate the task with the model
            #####
            # Sample parameters for your simulating model using their distributions
            # defined in the .parameters attribute of the model class
            simul_parameters = {
                ...
            }
            simulating_model = ...
            simulated_choices = ...

            #########################################################
            ### Fit all candidate models to the simulated data
            ####

            # Loop through all candidate models, fit them and get the AIC for each of them
            aics = []
            for candidate_model_class in model_classes:
                fitted_candidate_model = ...
                aic = ...
                aics.append(aic)

            #######################################################
            ### Select the winning model
            ####

            # Retrieve the index of the model with the best AIC
            select_idx = ...
            results.iloc[simul_idx, select_idx]  += 1

    return results


_ = perform_model_recovery(demo_task, [IntegrationModel, ExtremumDetectionModel], 2)
print("👌The code runs")

📝Run your `perform_model_recovery` function using the full `task` (**not** the demo_task), the two model classes, and 30 rounds (or less if your computer is a bit slow...)

In [ ]:
model_classes = [...]
model_recovery_results = perform_model_recovery(..., model_classes, n_rounds=...)

📝 Complete the `plot_model_recovery_inversion_matrix` function that plots the inversion matrix of the model recovery results.
> Note: Although in TD 7, we first computed the confusion matrix, it is not a necessary step. You can directly compute the inversion matrix by performing the appropriate normalization on the model recovery results.

In [ ]:
def plot_model_recovery_inversion_matrix(model_recovery_results: pd.DataFrame):
    # Compute the inversion matrix (you need to normalize the results in the right way)
    inversion_matrix = ...

    # Make a heatmap with appropriate title and axis labels
    ...
    ...
    plt.show()

Call your function to plot the inversion matrix of the model recovery results.

In [ ]:
plot_model_recovery_inversion_matrix(model_recovery_results)

## Participant Data Analysis
Now we will use data from simulated subjects and try to see what model best explains their behavior.

### Load and inspect the data
The data is given in a DataFrame. The index of each rows indicates a string ID of the participant. Then each column represents the decision taken by the participant at the end of every block.

There are **9 participants**, who completed **200 blocks**.

In [ ]:
data = pd.read_csv('data.csv', index_col=0)
data

## Fit the models to each subject

📝Define again the model classes that we want to fit to the data (here, it is the same two models as before).

In [ ]:
model_classes = [...]

📝Complete the `fit_all_models_to_each_subject` function to fit all models to each subject's data.

In [ ]:
def fit_all_models_to_each_subject(model_classes: list, task: SequentialPerceptionTask, data: pd.DataFrame):
    # Instantiate the results as a zero matrix with the subject ids as row index and model names as column labels
    results_aic = pd.DataFrame(index=data.index, columns=[mc.__name__ for mc in model_classes])

    # Iterate through the data row by row
    for subject_id, subject_choices in tqdm(data.iterrows()):
        # Reshape the subject choices as a column vector
        subject_choices = subject_choices.values.reshape(-1, 1)

        for model_class in model_classes:
            # TO DO: Fit the model to the subject's choices
            fitted_model = ...

            # TO DO: Compute the AIC for the fitted model
            aic = ...

            # TO DO: Update the result dataframe
            results_aic.loc[subject_id, model_class.__name__] = ...

    return results_aic

📝With the `fit_all_models_to_each_subject` function, fit all models to each subject's data and collect the AIC scores in a dataframe.

Examine the `aic_per_subject_and_model` dataframe to make sure it seems reasonable.

In [ ]:
aic_per_subject_and_model = ...
aic_per_subject_and_model

## Model Selection

### (Approximate) Log Model Evidence
When we do model selection, it is common to use the log model evidence of a model as a measure of how well it explains the data.
The log model evidence of a model $M$ on behavioral data $D$ is defined as the log probability of  $\log p(M|D)$.

We have already computed the *Akaike Information Criterion* (AIC) for each model and each subject, but this criterion does not have the same interpretation as the log model evidence. We need to rescale it to make it look like a log probability.

We will approximate the log model evidence using the AIC as follows:
$$
\log p(M|D) \approx -\frac{1}{2} \text{AIC}(M|D)
$$

📝Compute the `lme_per_subject` dataframe, which is the log model evidence for each model and each subject.

In [ ]:
lme_per_subject = ...
lme_per_subject

### Fixed effect model selection
Do you remember the difference between fixed effects and random effects model selection?

- **Fixed effect**: We assume that the model is the same for all participants. To compare models, we sum the log model evidence across all participants for each model.
- **Random effect**: We assume that the model is different for each participant. Here, we can use the log model evidence to estimate the probability that each model generated the data for each subject.

📝 Compute the log model evidence for each model by summing the log model evidences across participants.

`lme_fixed_effect` should be a Pandas Series that contains the log model evidence for each model.

In [ ]:
lme_fixed_effect = ...
lme_fixed_effect

📝 Show the comparison of the log model evidences for each model using a bar plot.


In [ ]:
# Your code here

💭 What model seems to best explain the behavior of the entire sample of subjects?

### Random effects model selection
In random effects model selection, we assume that the model is different for each participant, and the goal is to find the model that best explains the data for each participant.

To do this, we calculate the log model evidence for both models for each subject, subtract one from the other, and visualize the differences. This allows us to see which model fits each subject better.

📝Compute the difference between the log model evidence of the `IntegrationModel` and the `ExtremumDetectionModel` for each subject.

Store the result in a Pandas Series called `lme_diff_per_subject`, where the index is the subject IDs and the values are the differences.

In [ ]:
lme_diff_per_subject = ...
lme_diff_per_subject

📝Visualize the differences in a horizontal bar plot. The y axis should be the subject ids and the x axis should be the difference between the log model evidences.

In [ ]:
# Your code here

💭 Which subjects seem to operate according to the **integration model**? And **extremum detection model**?

### 💪 Cognitive Phenotyping (optional)
If you want to go futher, you could determine the **cognitive phenotype** of all the subject to whom the **integration model** is a best fit. Give the estimated parameter values for each of them.

In [ ]:
# Your code here

You can do the same with the subjects that are best fit by the **extremum detection model**.

In [ ]:
# Your code here